In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import joblib
import os
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_PATH = r'D:\S2\thesis\cond\project_ids'
DATA_DIR = os.path.join(BASE_PATH, 'data', 'processed')

# Load Data Laten
print("⏳ Loading Latent Data...")
latent_data = joblib.load(os.path.join(DATA_DIR, 'latent_features.pkl'))

X_train = torch.FloatTensor(latent_data['X_train']).to(device)
y_train = torch.LongTensor(latent_data['y_train']).to(device)
X_test = torch.FloatTensor(latent_data['X_test']).to(device)
y_test = torch.LongTensor(latent_data['y_test']).to(device)

# Buat DataLoader
train_ds = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True)

print(f"✅ Data Ready! Training samples: {len(X_train)}")

⏳ Loading Latent Data...
✅ Data Ready! Training samples: 1700000


In [13]:
class ENetV2_FinalPush(nn.Module):
    def __init__(self, input_dim=16, num_classes=34):
        super(ENetV2_FinalPush, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4), # Sedikit lebih tinggi untuk cegah overfitting
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.network(x)

model_final_push = ENetV2_FinalPush().to(device)

In [14]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss()

    def forward(self, inputs, targets):
        logpt = -self.ce(inputs, targets)
        pt = torch.exp(logpt)
        loss = self.alpha * (1 - pt)**self.gamma * -logpt
        return loss

criterion = FocalLoss(gamma=2) # Gamma=2 adalah sweet spot untuk data imbalanced
optimizer = optim.AdamW(model_final_push.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=5, factor=0.5, verbose=True)

c:\Users\Z\miniconda3\envs\tesis\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [15]:
# --- 4. TRAINING LOOP (FIXED LOGGING) ---
epochs_final = 100
best_acc = 0

print("🚀 Starting Final Push Training...")
for epoch in range(epochs_final):
    model_final_push.train()
    total_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model_final_push(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    # Validasi per epoch
    model_final_push.eval()
    with torch.no_grad():
        y_pred_out = model_final_push(X_test)
        y_pred = torch.argmax(y_pred_out, dim=1)
        current_acc = (y_pred == y_test).float().mean().item()
    
    # Ambil LR saat ini sebelum scheduler step (untuk monitoring manual)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Update Scheduler berdasarkan akurasi
    scheduler.step(current_acc)
    
    # Cek apakah LR berubah setelah scheduler.step
    new_lr = optimizer.param_groups[0]['lr']
    if new_lr < current_lr:
        print(f"📉 LR Adjusted: {current_lr:.6f} -> {new_lr:.6f}")
    
    # Simpan model jika dapet akurasi terbaik
    if current_acc > best_acc:
        best_acc = current_acc
        torch.save(model_final_push.state_dict(), 'enet_v2_ultra_best.pth')
        
    # Print progress setiap 5 epoch
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{epochs_final}] | Loss: {total_loss/len(train_loader):.4f} | Acc: {current_acc*100:.2f}% | LR: {new_lr:.6f} (Best: {best_acc*100:.2f}%)")

    # Target 85% - 86%
    if current_acc >= 0.86:
        print(f"🎯 Target Reached at Epoch {epoch+1}! Accuracy: {current_acc*100:.2f}%")
        break

print("-" * 45)
print(f"✅ TRAINING SELESAI!")
print(f"🏆 Akurasi Tertinggi di Test Set: {best_acc*100:.4f}%")
print("-" * 45)

🚀 Starting Final Push Training...
Epoch [5/100] | Loss: 0.2383 | Acc: 81.49% | LR: 0.001000 (Best: 81.49%)
Epoch [10/100] | Loss: 0.1971 | Acc: 79.22% | LR: 0.001000 (Best: 82.25%)
Epoch [15/100] | Loss: 0.1794 | Acc: 78.81% | LR: 0.001000 (Best: 82.50%)
Epoch [20/100] | Loss: 0.1696 | Acc: 79.26% | LR: 0.001000 (Best: 82.52%)
Epoch [25/100] | Loss: 0.1617 | Acc: 79.15% | LR: 0.001000 (Best: 82.75%)
Epoch [30/100] | Loss: 0.1561 | Acc: 82.32% | LR: 0.001000 (Best: 83.15%)
📉 LR Adjusted: 0.001000 -> 0.000500
Epoch [35/100] | Loss: 0.1438 | Acc: 82.77% | LR: 0.000500 (Best: 83.15%)
📉 LR Adjusted: 0.000500 -> 0.000250
Epoch [40/100] | Loss: 0.1396 | Acc: 82.52% | LR: 0.000250 (Best: 83.15%)
Epoch [45/100] | Loss: 0.1334 | Acc: 82.74% | LR: 0.000250 (Best: 83.15%)
📉 LR Adjusted: 0.000250 -> 0.000125
Epoch [50/100] | Loss: 0.1303 | Acc: 82.79% | LR: 0.000125 (Best: 83.15%)
📉 LR Adjusted: 0.000125 -> 0.000063
Epoch [55/100] | Loss: 0.1285 | Acc: 79.46% | LR: 0.000063 (Best: 83.15%)
📉 LR Adju

In [16]:
import torch
import os
import shutil

# 1. Setup Path
model_dir = r'D:\S2\thesis\cond\project_ids\models'
os.makedirs(model_dir, exist_ok=True)

# 2. Save Kondisi Terakhir (Epoch 100)
final_path = os.path.join(model_dir, 'enet_v2_baseline_final_100.pth')
torch.save(model_final_push.state_dict(), final_path)

# 3. Pindahkan Versi TERBAIK (83.15%)
best_source = 'enet_v2_ultra_best.pth'
best_target = os.path.join(model_dir, 'enet_v2_baseline_best_8315.pth')

if os.path.exists(best_source):
    shutil.move(best_source, best_target)
    print(f"🏆 BEST MODEL (83.15%) diamankan ke: {best_target}")
else:
    print("⚠️ File best otomatis tidak ditemukan, pastikan sudah tersimpan di memory.")

print(f"✅ Final State diamankan ke: {final_path}")

🏆 BEST MODEL (83.15%) diamankan ke: D:\S2\thesis\cond\project_ids\models\enet_v2_baseline_best_8315.pth
✅ Final State diamankan ke: D:\S2\thesis\cond\project_ids\models\enet_v2_baseline_final_100.pth
